# Will a UBC station still have an e-bike when I arrive?

Imagine Brock Commons shows one e-bike and you are five minutes away. Should the app flag the backup station? We simulate that decision every five minutes, at each campus station currently showing at least one e-bike. We predict the probability that **the station will report zero e-bikes or rentals closed at arrival**, separately for five- and ten-minute walks.

The CityBikes archive records station changes, chiefly changes in total bikes or empty docks. A swap between bike types may go unrecorded. These labels describe archived inventory and explicit rental closures. Successful unlocks, staff actions and individual bike movements would need other evidence.

Run the cells top to bottom after preparing the local Parquet files. The comparison uses three earlier validation months to choose each horizon's model and evaluates the frozen choices on the configured test month. The default is October 2026 UTC. Archive updates after that month leave this checkpoint reproducible.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from threadpoolctl import threadpool_limits

# These are pilot choices. Set them before inspecting the held-out month.
TEST_MONTH_UTC = "2026-10"
HORIZONS = (5, 10)
GRID_MIN = 5
MAX_AGE_MIN = 10       # most recent station report at decision time
NETWORK_GAP_MIN = 10   # largest allowed gap in citywide archive events
HISTORY_MIN = 15
WARNING_RATE = 0.20
TIE_MARGIN = 0.01      # prefer simpler models within 1 percentage point of the best precision
LOCAL_TZ = "America/Vancouver"

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "data" / "processed" / "ubc_inventory.parquet").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Prepare data/processed/ubc_inventory.parquet first. See the repository README.")
DATA = ROOT / "data" / "processed"
inventory = pd.read_parquet(DATA / "ubc_inventory.parquet")
network = pd.read_parquet(DATA / "network_observations.parquet", columns=["observed_at"])
inventory["observed_at"] = pd.to_datetime(inventory["observed_at"], utc=True)
network = pd.DatetimeIndex(pd.to_datetime(network["observed_at"], utc=True)).dropna().unique().sort_values()
test_month = pd.Period(TEST_MONTH_UTC, freq="M").start_time.tz_localize("UTC")
archive_end = test_month + pd.offsets.MonthBegin(1)
inventory = inventory.loc[inventory["observed_at"] < archive_end].copy()
network = network[network < archive_end]
assert len(network) > 1, "The citywide archive coverage index is empty."
assert inventory["station_id"].notna().all() and inventory["observed_at"].notna().all()

inventory["station_id"] = inventory["station_id"].astype(str)
inventory = inventory.sort_values(["station_id", "observed_at"])
# Ambiguous simultaneous states cannot define which count a rider saw.
duplicates = inventory.duplicated(["station_id", "observed_at"], keep=False)
if duplicates.any():
    same_time = inventory.loc[duplicates].groupby(["station_id", "observed_at"])
    fields = ["ebikes", "total_bikes", "is_renting"]
    if same_time[fields].nunique(dropna=False).gt(1).any().any():
        raise ValueError("The input contains conflicting station states at the same timestamp.")
    inventory = inventory.drop_duplicates(["station_id", "observed_at"])

print(f"{len(inventory):,} station changes at {inventory.station_id.nunique()} hubs")
print(f"Archive coverage index: {network.min()} to {network.max()}")
display(inventory.groupby("station_name", dropna=False).agg(
    changes=("station_id", "size"), typed_counts=("ebikes", "count"),
    rental_status_known=("is_renting", "count"), first=("observed_at", "min"), last=("observed_at", "max")
))

## Reconstruct decisions and arrivals

For each grid time we take the last archived station record **at or before** that time. A decision enters the pilot only when the record is at most ten minutes old, has a valid positive e-bike count and permits rentals or has unreported rental status. Explicit rental closure at arrival counts as unavailable. An open or unknown-status station needs a known bike count to form a label.

We keep a shared set of decisions for both walks. No citywide event gap may exceed ten minutes from the start of the history window through the longer arrival. We require events bracketing that entire interval. At arrival, a quiet station keeps its last state. The decision freshness rule limits that carried state to twenty minutes at the longest horizon. This avoids selectively dropping stations whose stock happened to stay constant. We report how often a newer station record actually arrived. The evaluation therefore applies to recently updated stations during periods of citywide activity. Quiet hours and older readings require separate evidence.

Elapsed times and sampling use UTC. Clock and weekday features use Vancouver local time. Inventory changes are **net changes** that may include both riders and staff.

In [ ]:
SECOND = 1_000_000_000
MINUTE = 60 * SECOND

def nanoseconds(times):
    return pd.DatetimeIndex(times).as_unit("ns").asi8

net = nanoseconds(network)
segment = np.cumsum(np.r_[False, np.diff(net) > NETWORK_GAP_MIN * MINUTE])
grid = nanoseconds(pd.date_range(network.min().ceil(f"{GRID_MIN}min"),
                                 network.max().floor(f"{GRID_MIN}min"), freq=f"{GRID_MIN}min"))
first = np.searchsorted(net, grid - HISTORY_MIN * MINUTE, side="right") - 1
last = np.searchsorted(net, grid + max(HORIZONS) * MINUTE, side="left")
covered = (first >= 0) & (last < len(net))
covered[covered] &= segment[first[covered]] == segment[last[covered]]
grid = grid[covered]

def station_examples(group):
    times = nanoseconds(group.observed_at)
    count = group.ebikes.to_numpy(dtype=float, na_value=np.nan, copy=True)
    total = group.total_bikes.to_numpy(dtype=float, na_value=np.nan, copy=True)
    count[(count < 0) | ((total >= 0) & (count > total))] = np.nan
    total[total < 0] = np.nan
    closed = group.is_renting.eq(False).fillna(False).to_numpy(dtype=bool)

    now = np.searchsorted(times, grid, side="right") - 1
    age = (grid - times[now]) / MINUTE
    eligible = (now >= 0) & (age <= MAX_AGE_MIN) & (count[now] > 0) & ~closed[now]
    decision, now = grid[eligible], now[eligible]
    frame = pd.DataFrame({
        "time": pd.to_datetime(decision, utc=True), "station_id": group.station_id.iloc[0],
        "ebikes": count[now], "total_bikes": total[now], "age_min": age[eligible],
    })
    for lag in (5, HISTORY_MIN):
        past_time = decision - lag * MINUTE
        past = np.searchsorted(times, past_time, side="right") - 1
        known = (past >= 0) & (past_time - times[past] <= MAX_AGE_MIN * MINUTE) & ~closed[past]
        # A pre-outage carry at the left edge of the history window is unknown.
        past_net = np.searchsorted(net, times[past], side="right") - 1
        decision_net = np.searchsorted(net, decision, side="right") - 1
        known &= (past_net >= 0) & (segment[past_net] == segment[decision_net])
        frame[f"net_change_{lag}"] = np.where(known, count[now] - count[past], np.nan)
    frame["reports_15"] = now - (np.searchsorted(times, decision - HISTORY_MIN * MINUTE, side="right") - 1)
    for minutes in HORIZONS:
        arrival = np.searchsorted(times, decision + minutes * MINUTE, side="right") - 1
        known = closed[arrival] | np.isfinite(count[arrival])
        empty = closed[arrival] | (count[arrival] == 0)
        frame[f"empty_{minutes}"] = np.where(known, empty.astype(float), np.nan)
        frame[f"new_report_{minutes}"] = arrival != now
    return frame

examples = pd.concat([station_examples(g) for _, g in inventory.groupby("station_id")], ignore_index=True)
target_cols = [f"empty_{h}" for h in HORIZONS]
eligible_decisions = len(examples)
examples = examples.dropna(subset=target_cols).sort_values("time").reset_index(drop=True)
if examples.empty:
    raise ValueError("No paired outcomes passed coverage, freshness and inventory checks.")
local = examples.time.dt.tz_convert(LOCAL_TZ)
clock_angle = 2 * np.pi * (local.dt.hour + local.dt.minute / 60) / 24
examples["hour_sin"], examples["hour_cos"] = np.sin(clock_angle), np.cos(clock_angle)
examples["weekend"] = (local.dt.dayofweek >= 5).astype(int)
print(f"{len(examples):,} complete paired decisions out of {eligible_decisions:,} eligible decisions")
display(pd.DataFrame({
    "horizon_min": HORIZONS,
    "new_station_report_by_arrival": [examples[f"new_report_{h}"].mean() for h in HORIZONS],
}))

## Compare models at the same warning rate

The count-only baseline estimates the empty-arrival rate for each starting count. All counts of five or more share one bucket, and each bucket is smoothed with ten decisions at the training prevalence. This makes a stronger comparison than always predicting that a bike will remain.

The context logistic model adds hub, Vancouver clock time, weekend and total bikes. The history logistic model also sees net e-bike changes over five and fifteen minutes, recent report count and age of the latest report. Comparing the two estimates what the temporal features add. A shallow boosted tree uses those same inputs and can learn interactions such as how a recent drop matters more when one bike is left.

At each horizon, we train on all earlier eligible decisions. The three validation months are scored separately. For the final test month we refit using all earlier data. Any decision whose outcome would reach a time boundary is discarded from the earlier period. We require at least 90 days of earlier decision history and use only evaluation months with network events on at least 90% of UTC dates and within one day of each end.

In [ ]:
def utc_month(times):
    return pd.DatetimeIndex(times).tz_convert("UTC").tz_localize(None).to_period("M")

network_days = pd.DatetimeIndex(network.normalize().unique())
candidate_months = []
for month in sorted(utc_month(network_days).unique()):
    start = month.start_time.tz_localize("UTC")
    end = (month + 1).start_time.tz_localize("UTC")
    days = network_days[(network_days >= start) & (network_days < end)]
    if (len(days) >= 0.9 * month.days_in_month and days.min() == start
            and days.max() == end - pd.Timedelta(days=1)
            and start - examples.time.min() >= pd.Timedelta(days=90)):
        candidate_months.append(start)
if test_month not in candidate_months:
    raise ValueError(f"Configured test month {TEST_MONTH_UTC} lacks sufficient archive coverage or prior history.")
validation_months = [m for m in candidate_months if m < test_month][-3:]
if len(validation_months) < 3:
    raise ValueError("Need three eligible validation months before the configured test month.")
print("Validation:", [m.strftime("%Y-%m") for m in validation_months], "  Final test:", test_month.strftime("%Y-%m"))

def split_at(start):
    end = start + pd.offsets.MonthBegin(1)
    arrival = examples.time + pd.Timedelta(minutes=max(HORIZONS))
    train = examples.loc[arrival < start]
    evaluation = examples.loc[(examples.time >= start) & (arrival < end)]
    if train.empty or evaluation.empty:
        raise ValueError(f"No eligible training or evaluation decisions for {start:%Y-%m}.")
    return train, evaluation

In [ ]:
CONTEXT = ["ebikes", "total_bikes", "hour_sin", "hour_cos", "weekend"]
HISTORY = CONTEXT + ["net_change_5", "net_change_15", "reports_15", "age_min"]
MODELS = ["Count only", "Logistic context", "Logistic history", "Boosted history"]

def predict(model, train, evaluation, horizon):
    target = f"empty_{horizon}"
    y = train[target].astype(int)
    if model == "Count only":
        bucket = train.ebikes.clip(upper=5)
        stats = y.groupby(bucket).agg(["sum", "count"])
        rate = (stats["sum"] + 10 * y.mean()) / (stats["count"] + 10)
        return evaluation.ebikes.clip(upper=5).map(rate).fillna(y.mean()).to_numpy()

    if y.nunique() < 2:
        raise ValueError(f"Training data has only one outcome at {horizon} minutes.")
    numeric = CONTEXT if model == "Logistic context" else HISTORY
    columns = numeric + ["station_id"]
    numbers = make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler())
    preprocessor = ColumnTransformer([
        ("numeric", numbers, numeric),
        ("hub", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["station_id"]),
    ])
    if model == "Boosted history":
        estimator = HistGradientBoostingClassifier(max_depth=3, max_iter=80, learning_rate=0.07,
                                                  min_samples_leaf=40, l2_regularization=1,
                                                  early_stopping=False, random_state=42)
    else:
        estimator = LogisticRegression(max_iter=400)
    pipeline = make_pipeline(preprocessor, estimator)
    with threadpool_limits(limits=2):
        pipeline.fit(train[columns], y)
        return pipeline.predict_proba(evaluation[columns])[:, 1]

Every model gets **20 warnings per 100 eligible walks**. Precision answers “of the warnings we issued, how many walks ended at an empty station?” Recall measures the share of all empty arrivals warned about. Brier score measures the accuracy of the full probability forecast, where lower is better. Average precision (AP) summarizes ranking across budgets.

When scores tie at the budget boundary, each tied decision gets the same fractional warning credit. This is the expected result of fair random tie-breaking. A count-only model cannot earn credit by picking a favorable order within a block of one-bike stations. This budget compares rankings offline. A live warning cutoff would need to be set from earlier data.

In [ ]:
def warning_weights(score, rate=WARNING_RATE):
    score = np.asarray(score, dtype=float)
    if not len(score) or not np.isfinite(score).all():
        raise ValueError("Need at least one finite score for every decision.")
    budget = rate * len(score)
    boundary = np.partition(score, -int(np.ceil(budget)))[-int(np.ceil(budget))]
    above, ties = score > boundary, score == boundary
    return above.astype(float) + ties * ((budget - above.sum()) / ties.sum())

def metrics(y, score):
    y = np.asarray(y, dtype=int)
    w = warning_weights(score)
    hits = np.dot(w, y)
    return {
        "walks": len(y), "empty_arrivals": y.sum(), "empty_rate": y.mean(),
        "warning_precision": hits / w.sum(), "empty_recall": hits / y.sum() if y.sum() else np.nan,
        "AP": average_precision_score(y, score) if y.sum() else np.nan,
        "Brier": brier_score_loss(y, score),
    }

# Boundary ties receive equal credit even when every forecast is identical.
assert np.allclose(warning_weights([0.5] * 10), WARNING_RATE)

In [ ]:
validation_rows = []
for start in validation_months:
    train, validation = split_at(start)
    for horizon in HORIZONS:
        y = validation[f"empty_{horizon}"]
        for model in MODELS:
            score = predict(model, train, validation, horizon)
            validation_rows.append({"month": start.strftime("%Y-%m"), "horizon": horizon, "model": model,
                                    **metrics(y, score)})

validation_results = pd.DataFrame(validation_rows)
monthly_precision = validation_results.pivot(index=["horizon", "model"], columns="month", values="warning_precision")
print("Warning precision by validation month:")
display(monthly_precision.round(3))
means = validation_results.groupby(["horizon", "model"])[["warning_precision", "empty_recall", "AP", "Brier"]].mean()
display(means.round(3))
# Equal weight per month; model ordering above breaks near ties in favor of simplicity.
selected = {}
for horizon in HORIZONS:
    precision = means.loc[horizon, "warning_precision"].reindex(MODELS)
    selected[horizon] = precision[precision >= precision.max() - TIE_MARGIN].index[0]
print("Selection frozen from validation:", selected)

## Check the frozen choices on a later month

The first table compares the chosen model against count-only on all eligible walks. The second repeats that comparison among **one-bike decisions**, giving each model the same warning budget within that subgroup. That isolates whether context distinguishes riders who see precisely the same count. All counts below are hypothetical decision opportunities sampled every five minutes. One stockout can affect multiple opportunities.

The final table checks how widely the result varies when we resample whole Vancouver calendar days together. It reports a paired 95% bootstrap interval for incremental warning precision versus count-only, using the month's warning assignments. The interval captures variation between days. Adjacent days can still be correlated.

In [ ]:
train, test = split_at(test_month)
test_rows, day_results = [], []
rng = np.random.default_rng(42)

def paired_day_interval(y, scores, base_scores, days):
    # Aggregate warning counts and correctly warned empty arrivals before resampling days.
    w, b = warning_weights(scores), warning_weights(base_scores)
    by_day = pd.DataFrame({"mw": w, "mh": w * y, "bw": b, "bh": b * y, "day": days}).groupby("day").sum()
    draws = rng.integers(len(by_day), size=(1000, len(by_day)))
    sums = by_day.to_numpy()[draws].sum(axis=1)
    ok = (sums[:, 0] > 0) & (sums[:, 2] > 0)
    delta = sums[ok, 1] / sums[ok, 0] - sums[ok, 3] / sums[ok, 2]
    return np.quantile(delta, [0.025, 0.975]) if len(delta) else [np.nan, np.nan]

for horizon, choice in selected.items():
    forecasts = {model: predict(model, train, test, horizon) for model in dict.fromkeys(["Count only", choice])}
    for subgroup, mask in [("all", np.ones(len(test), dtype=bool)), ("one bike", test.ebikes.eq(1).to_numpy())]:
        if not mask.any():
            continue
        y = test[f"empty_{horizon}"].to_numpy(dtype=int)[mask]
        days = test.time.dt.tz_convert(LOCAL_TZ).dt.date.to_numpy()[mask]
        for model, forecast in forecasts.items():
            test_rows.append({"horizon": horizon, "sample": subgroup, "model": model, **metrics(y, forecast[mask])})
        model_score, base_score = forecasts[choice][mask], forecasts["Count only"][mask]
        delta = metrics(y, model_score)["warning_precision"] - metrics(y, base_score)["warning_precision"]
        low, high = paired_day_interval(y, model_score, base_score, days)
        day_results.append({"horizon": horizon, "sample": subgroup, "chosen": choice,
                            "precision_gain": delta, "day_bootstrap_low": low, "day_bootstrap_high": high})

test_results = pd.DataFrame(test_rows).set_index(["sample", "horizon", "model"])
print(f"Final test month {test_month:%Y-%m} — {test.station_id.nunique()} stations on {len(test):,} decisions")
for subgroup in test_results.index.get_level_values("sample").unique():
    display(test_results.loc[subgroup].round(3))
display(pd.DataFrame(day_results).round(3))

## Decision for the pilot

A gain of 0.05 in warning precision means **five extra empty arrivals identified per 100 warnings** at the same warning volume. Check whether the gain survives in the final month and on one-bike walks. Compare the month-to-month validation rows as well. If count-only wins, a simple count-based warning earns the pilot. If the held-out gains are small or inconsistent, this archive gives little reason to add model complexity.

Even a correct warning only helps when a reachable backup has a bike. These scores establish whether the app could identify risky walks. Measuring avoided wasted walks would require testing the backup suggestion with riders and observing whether they could actually unlock a bike.